# TIR-JDE — model karşılaştırma pipeline'ı (Colab)

Strong Baseline ve Dist-Tracker'ı **aynı veri, aynı split ve aynı değerlendirme** ile eğitir, test eder ve
her şeyi MLflow'a (DagsHub) kaydeder.

**Önce:** *Runtime → Change runtime type → A100* (Strong Baseline'ın flash-attn'i Ampere ister).
**Gizli anahtarlar** (soldaki 🔑 Secrets): `DAGSHUB_USER`, `DAGSHUB_TOKEN`. Tanımlı değilse kayıtlar Drive'daki
`mlflow.db` dosyasına yerel olarak yazılır.

Akış: kurulum → veri → doğrulama grubu → preprocess → duman testi (1 epoch) → tam eğitim (3 seed) → karşılaştırma.

In [ ]:
# 1) Drive ve klasörler
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/tir-jde-work'          # ağırlıklar, çıktılar, yerel MLflow kaydı
BSB_ROOT = '/content/drive/MyDrive/datasets/BSB'       # ham BSB verisi
os.makedirs(WORK, exist_ok=True)
os.environ.update(TIRJDE_WORK=WORK, BSB_ROOT=BSB_ROOT, TIRJDE_BASELINES='/content/baselines')

try:  # DagsHub (MLflow) erişimi Colab Secrets'tan
    from google.colab import userdata
    os.environ['DAGSHUB_USER'] = userdata.get('DAGSHUB_USER')
    os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
    print('MLflow -> DagsHub')
except Exception:
    print('DagsHub anahtarı yok: MLflow kayıtları', WORK + '/mlflow.db', 'altına yazılacak')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) TIR-JDE ve pipeline bağımlılıkları (Colab'ın kendi Python'u)
%cd /content
!test -d TIR-JDE && git -C TIR-JDE pull -q || git clone -q https://github.com/silaylc/TIR-JDE.git
%cd /content/TIR-JDE
!pip install -q mlflow pyyaml loguru python-dotenv tqdm typer opencv-python-headless matplotlib scipy pandas \
    "git+https://github.com/JonathonLuiten/TrackEval.git" && pip install -q -e . --no-deps
!git log -1 --format='TIR-JDE %h %s'

In [ ]:
# 3) Baseline depoları (configs/model/*.yaml'daki commit'lere sabitlenir)
import yaml, subprocess
for name in ('strong_baseline', 'dist_tracker'):
    m = yaml.safe_load(open(f'configs/model/{name}.yaml'))['model']
    dest = f"/content/baselines/{name}"
    if not os.path.isdir(dest):
        subprocess.run(['git', 'clone', '-q', m['repo'], dest], check=True)
    subprocess.run(['git', '-C', dest, 'checkout', '-q', m['commit']], check=True)
    print(name, subprocess.run(['git', '-C', dest, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)

In [ ]:
# 4) Strong Baseline ortamı: Python 3.11 + torch 2.2.2 (CUDA 11.8) + flash-attn 2.7.3 (yazarın README'si)
!pip install -q uv
!uv python install 3.11 3.8
!uv venv /content/envs/sb --python 3.11 --seed --clear -q
SB = '/content/envs/sb/bin/python'
!uv pip install -q --python {SB} torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu118
%cd /content/baselines/strong_baseline/BoT-SORT/yolov12
!wget -q -nc https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!uv pip install -q --python {SB} -r requirements.txt
%cd ..
!uv pip install -q --python {SB} -r requirements.txt ultralytics==8.3.88 cython faiss-cpu seaborn
!uv pip install -q --python {SB} --no-build-isolation cython_bbox "git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI"
!{SB} -c "import torch, flash_attn; print('SB:', torch.__version__, torch.version.cuda, 'flash-attn', flash_attn.__version__)"
os.environ['SB_PYTHON'] = SB
%cd /content/TIR-JDE

In [ ]:
# 5) Dist-Tracker ortamı: Python 3.8 + torch 2.4.1 (yazarın environment.yml'i, pip'teki 'ultralytics' HARİÇ)
!uv venv /content/envs/dist --python 3.8 --seed --clear -q
DIST = '/content/envs/dist/bin/python'
!uv pip install -q --python {DIST} torch==2.4.1 torchvision==0.19.1 numpy==1.24.4 opencv-python==4.11.0.86 \
    pillow==10.4.0 pyyaml==6.0.2 requests==2.32.3 scipy==1.10.1 matplotlib==3.7.5 pandas==2.0.3 seaborn==0.13.2 \
    tqdm==4.67.1 psutil==7.0.0 py-cpuinfo==9.0.0 ultralytics-thop==2.0.14 lap==0.5.12
# Yazarın fork'u yerine pip paketi yüklenirse NWD/SSQ kaybı sessizce devre dışı kalır:
!{DIST} -m pip uninstall -y -q ultralytics 2>/dev/null; echo "pip ultralytics yok: OK"
!{DIST} -c "import torch; print('Dist:', torch.__version__, torch.cuda.is_available())"
os.environ['DIST_PYTHON'] = DIST

## Veri
BSB eğitim verisi: [Zenodo 15853476](https://doi.org/10.5281/zenodo.15853476). Drive'da `BSB_ROOT` altına açın ve
klasör adlarını `configs/data/bsb.yaml` ile eşleştirin (`videos_dir`, `labels_dir` veya hazır kareler için `frames_dir`).
Aşağıdaki hücre klasör yapısını gösterir.

In [ ]:
# 6) Ham verinin yapısını kontrol et
!ls $BSB_ROOT | head; echo; ls $BSB_ROOT/* | head -n 8
!head -n 3 data/splits/bsb_train.txt 2>/dev/null || echo "data/splits/bsb_train.txt yok: 102 BSB eğitim sekansının adlarını bu dosyaya yazın"

In [ ]:
# 7) Doğrulama grubu (BİR KEZ). Her aileden 1 sekans, sabit seed. Oluşan data/splits/val.txt dosyasını
#    TIR-JDE reposuna commit edin; sonraki tüm deneyler aynı grubu kullanır.
!test -f data/splits/val.txt && cat data/splits/val.txt || python -m tir_jde.pipeline make-val strong_baseline_bsb --per-family 1 --seed 0

In [ ]:
# 8) Preprocess: YOLO etiketleri, ReID kırpıntıları, takip girdisi, GT (split hash'ine göre önbelleğe alınır)
!python -m tir_jde.pipeline preprocess strong_baseline_bsb

In [ ]:
# 9) Duman testi: her model 1 epoch, çıktıların ve MLflow kaydının uçtan uca çalıştığını doğrular
!python -m tir_jde.pipeline run dist_tracker_bsb   tag=smoke model.detector.epochs=1
!python -m tir_jde.pipeline run strong_baseline_bsb tag=smoke model.detector.epochs=1

In [ ]:
# 10) Tam eğitim: 3 seed × 2 model (uzun sürer; Colab kapanırsa aynı hücreyi tekrar çalıştırın,
#     bitmiş seed'ler results.json'dan tanınır ve atlanır)
import pathlib
for exp in ('dist_tracker_bsb', 'strong_baseline_bsb'):
    for seed in (0, 1, 2):
        if pathlib.Path(f'{WORK}/runs/{exp}-s{seed}/results.json').exists():
            print('tamam:', exp, seed); continue
        !python -m tir_jde.pipeline run {exp} seed={seed}

In [ ]:
# 11) Karşılaştırma tablosu ve grafik (ortalama ± std, 3 seed)
!python -m tir_jde.pipeline compare strong_baseline_bsb dist_tracker_bsb --seeds 0 1 2 --out {WORK}/comparison
from IPython.display import Image, display
display(Image(f'{WORK}/comparison/comparison.png'))